# Deep Learning - CNN (MobileNetV2)

In [1]:
import numpy as np
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================================
# Configuration
# ============================================================================
print("="*60)
print("Deep Learning - CNN (MobileNetV2)")
print("="*60)

print("\n[1/6] Chargement des images d'entrainement...")

# Generateur pour l'entrainement AVEC augmentation
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
    )

# Chargement des donnees
training_set = train_datagen.flow_from_directory(
    './dataset/training_set',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=True
    )
print(f"✓ Training samples: {training_set.samples}")

print("\n[2/6] Chargement des images de test...")

# Generateur pour le test SANS augmentation
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

test_set = test_datagen.flow_from_directory(
    './dataset/test_set',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
    )

Deep Learning - CNN (MobileNetV2)

[1/6] Chargement des images d'entrainement...
Found 8000 images belonging to 2 classes.
✓ Training samples: 8000

[2/6] Chargement des images de test...
Found 2000 images belonging to 2 classes.


In [2]:
print("\n[3/6] Construction du modele MobileNetV2...")

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
    )
base_model.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)
model.summary()
print("✓ Modele construit")


[3/6] Construction du modele MobileNetV2...
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

✓ Modele construit


In [3]:
print("\n[4/6] Compilation du modele...")

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
    )

print("✓ Modele compile")


[4/6] Compilation du modele...
✓ Modele compile


In [4]:
print("\n[5/6] Entrainement du modele...")

history = model.fit(
    training_set,
    steps_per_epoch=training_set.samples // 32,
    epochs=10,
    validation_data=test_set,
    validation_steps=test_set.samples // 32,
    verbose=1
    )

print("✓ Entrainement termine")


[5/6] Entrainement du modele...
Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 188s 742ms/step - accuracy: 0.9246 - loss: 0.1907 - val_accuracy: 0.9829 - val_loss: 0.0540
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 72s 290ms/step - accuracy: 0.9664 - loss: 0.0873 - val_accuracy: 0.9829 - val_loss: 0.0423
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 83s 330ms/step - accuracy: 0.9737 - loss: 0.0722 - val_accuracy: 0.9854 - val_loss: 0.0363
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 83s 332ms/step - accuracy: 0.9765 - loss: 0.0663 - val_accuracy: 0.9884 - val_loss: 0.0339
Epoch 5/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 78s 313ms/step - accuracy: 0.9768 - loss: 0.0641 - val_accuracy: 0.9864 - val_loss: 0.0328
Epoch 6/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 73s 291ms/step - accuracy: 0.9779 - loss: 0.0561 - val_accuracy: 0.9879 - val_loss: 0.0326
Epoch 7/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 73s 290ms/step - accuracy: 0.9769 - loss: 0.0585 - val_accuracy: 0.9874 - val_loss: 0.0327
Epoch 8/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 74s 296ms

In [5]:
print("\n[6/6] Evaluation sur le test set...")

test_set.reset()
y_pred_proba = model.predict(test_set, steps=test_set.samples // 32 + 1)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()
y_true = test_set.classes[:len(y_pred)]
accuracy = accuracy_score(y_true, y_pred)

print(f"\n{'='*60}")
print(f"ACCURACY CNN : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"{'='*60}")

# Matrice de confusion
cm = confusion_matrix(y_true, y_pred)
print("\nMatrice de Confusion:")
print(cm)

# Classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Cats', 'Dogs']))


[6/6] Evaluation sur le test set...
63/63 ━━━━━━━━━━━━━━━━━━━━ 8s 122ms/step

ACCURACY CNN : 0.9870 (98.70%)

Matrice de Confusion:
[[976  24]
 [  2 998]]

Classification Report:
              precision    recall  f1-score   support

        Cats       1.00      0.98      0.99      1000
        Dogs       0.98      1.00      0.99      1000

    accuracy                           0.99      2000
   macro avg       0.99      0.99      0.99      2000
weighted avg       0.99      0.99      0.99      2000

